<a href="https://colab.research.google.com/github/EngenhariaSoftwarePUCRS/Inteligencia_Artificial/blob/develop/Trabalho01/Trabalho01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [145]:
integrantes = ["Augusto Baldino", "Felipe Freitas", "Isabela Kuser", "Luiza Heller", "Maria Eduarda Maia", "Paola Lopes"]
useful_links = [
    "https://pandas.pydata.org/pandas-docs/version/1.0/user_guide/style.html",
    "https://www.geeksforgeeks.org/how-to-replace-values-in-column-based-on-condition-in-pandas/",
    "https://www.geeksforgeeks.org/how-to-add-header-row-to-a-pandas-dataframe/",
    "https://discuss.streamlit.io/t/is-it-possible-to-center-data-on-cell-like-excel/32045/7",
]

In [146]:
import pandas as pd
from matplotlib import pyplot
from seaborn.palettes import mpl_palette as colors
from sklearn import metrics, model_selection, neighbors
from typing import Literal

In [147]:
targetClass = "Outcome"
columnNames = [
    "Top-Left", "Top-Middle", "Top-Right",
    "Middle-Left", "Middle-Middle", "Middle-Right",
    "Bottom-Left", "Bottom-Middle", "Bottom-Right",
    targetClass,
]
dataset = pd.read_csv(
    "tic-tac-toe.data",
    names=columnNames,
)
dataLinesCount = 958

In [148]:
X_VALUE = 1
B_VALUE = 0
O_VALUE = -1

def cell_to_number(cell: Literal['b', 'x', 'o']):
  if cell == 'b':
    return B_VALUE

  if cell == 'x':
    return X_VALUE

  if cell == 'o':
    return O_VALUE

  return cell

In [149]:
def number_to_cell(number: Literal[1, 0, -1]):
    if number == B_VALUE:
        return 'b'

    if number == X_VALUE:
        return 'x'

    if number == O_VALUE:
        return 'o'

    return number

In [150]:
def o_has_won(board: list[str]) -> bool:
    if len(board) != 9:
        raise ValueError("Board cannot have any different number of rows x columns than 9")

    for i in range(0, 9, 3):
        line = board[i:i+3]
        # print("Line: ", line)
        if all(cell == O_VALUE for cell in line):
            return True

    for i in range(0, 3):
        column = [board[i], board[i+3], board[i+6]]
        # print("Column: ", column)
        if all(cell == O_VALUE for cell in column):
            return True

    if board[4] != O_VALUE:
        return False

    diagonal1 = [board[0], board[8]]
    # print("D1: ", diagonal1)
    if all(cell == O_VALUE for cell in diagonal1):
        return True

    diagonal2 = [board[2], board[6]]
    # print("D2: ", diagonal2)
    if all(cell == O_VALUE for cell in diagonal2):
        return True

    return False

In [151]:
dataset.head(5)

,Top-Left,Top-Middle,Top-Right,Middle-Left,Middle-Middle,Middle-Right,Bottom-Left,Bottom-Middle,Bottom-Right,Outcome
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive
3,x,x,x,x,o,o,o,b,b,positive
4,x,x,x,x,o,o,b,o,b,positive


In [152]:
for columnName in columnNames:
  dataset[columnName] = dataset[columnName].apply(cell_to_number)

In [153]:
dataset.head(5)

,Top-Left,Top-Middle,Top-Right,Middle-Left,Middle-Middle,Middle-Right,Bottom-Left,Bottom-Middle,Bottom-Right,Outcome
0,1,1,1,1,-1,-1,1,-1,-1,positive
1,1,1,1,1,-1,-1,-1,1,-1,positive
2,1,1,1,1,-1,-1,-1,-1,1,positive
3,1,1,1,1,-1,-1,-1,0,0,positive
4,1,1,1,1,-1,-1,0,-1,0,positive


In [154]:
# NOTE: Run only once
for index, row in dataset.iterrows():
    board = row.array.tolist()[:-1]

    if row[targetClass] == 'positive':
        targetValue = "X Ganhou"
    elif o_has_won(board):
        targetValue = "O Ganhou"
    elif B_VALUE in board:
        targetValue = "Tem Jogo"
    else:
        targetValue = "Deu Velha"

    dataset.at[index, targetClass] = targetValue

In [155]:
x_ganhou_count = dataset[targetClass].eq('X Ganhou').sum()
o_ganhou_count = dataset[targetClass].eq('O Ganhou').sum()
deu_velha_count = dataset[targetClass].eq('Deu Velha').sum()
tem_jogo_count = dataset[targetClass].eq('Tem Jogo').sum()

# Define the outcomes and their counts
outcomes = {
    'X Ganhou': x_ganhou_count,
    'O Ganhou': o_ganhou_count,
    'Deu Velha': deu_velha_count,
    'Tem Jogo': tem_jogo_count,
}

print(outcomes)

{'X Ganhou': 626, 'O Ganhou': 316, 'Deu Velha': 16, 'Tem Jogo': 0}


In [156]:
# Define the features (X) and the target variable (outcome)
X = dataset.drop(columns=[targetClass])
y = dataset[targetClass]

# Define the split ratios
split_ratios = {'train': 0.7, 'validation': 0.15, 'test': 0.15}
validation_plus_test_ratio = split_ratios['validation'] + split_ratios['test']
test_to_validation_ratio = split_ratios['test'] / validation_plus_test_ratio

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = model_selection.train_test_split(X, y, train_size=split_ratios['train'], stratify=y, random_state=42)
X_validation, X_test, y_validation, y_test = model_selection.train_test_split(X_temp, y_temp, test_size=test_to_validation_ratio, stratify=y_temp, random_state=42)

# Print the shapes of the resulting sets to verify the split
# Print the distribution of outcomes in each set
print("Training set shape:", X_train.shape)
print("Training set distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation set shape:", X_validation.shape)
print("Validation set distribution:")
print(y_validation.value_counts(normalize=True))

print("\nTest set shape:", X_test.shape)
print("Test set distribution:")
print(y_test.value_counts(normalize=True))

Training set shape: (670, 9)
Training set distribution:
Outcome
X Ganhou     0.653731
O Ganhou     0.329851
Deu Velha    0.016418
Name: proportion, dtype: float64

Validation set shape: (144, 9)
Validation set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.333333
Deu Velha    0.013889
Name: proportion, dtype: float64

Test set shape: (144, 9)
Test set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.326389
Deu Velha    0.020833
Name: proportion, dtype: float64


In [157]:
kNN_classifier = neighbors.KNeighborsClassifier(n_neighbors=5)
kNN_classifier.fit(X_train, y_train)
predictions_test = kNN_classifier.predict(X_test)
print("k-NN accuracy: ", metrics.accuracy_score(y_test, predictions_test))

k-NN accuracy:  0.9722222222222222


In [178]:
def print_board(board):
    def cell_pretty(cell_value):
        cell = number_to_cell(cell_value)
        if cell == 'b':
            cell = ' '
        return cell

    for i in range(0, 9, 3):
        row_raw = board[i:i+3]
        row_str = [cell_pretty(cell_value) for cell_value in row_raw]
        row_pretty = " | ".join(row_str)
        print(row_pretty)

In [179]:
for i in range(len(X_validation)):
    print("Linha", i, "Tabuleiro:")
    print_board(X_validation.iloc[i].values)
    print("Outcome: ", y_validation.iloc[i], end="\n\n")

Linha 0 Tabuleiro:
x |   |  
x | x | o
o | o | x
Outcome:  X Ganhou

Linha 1 Tabuleiro:
x | x | x
  | x | o
o |   | o
Outcome:  X Ganhou

Linha 2 Tabuleiro:
x |   |  
x |   | o
x |   | o
Outcome:  X Ganhou

Linha 3 Tabuleiro:
x | o | x
  | o | x
o |   | x
Outcome:  X Ganhou

Linha 4 Tabuleiro:
x | x | x
o |   | x
  | o | o
Outcome:  X Ganhou

Linha 5 Tabuleiro:
o | x |  
o | o | x
x | x | o
Outcome:  O Ganhou

Linha 6 Tabuleiro:
o | o | o
x |   | x
  | x |  
Outcome:  O Ganhou

Linha 7 Tabuleiro:
x | o | o
x |   |  
x | x | o
Outcome:  X Ganhou

Linha 8 Tabuleiro:
x |   | o
x | x |  
x | o | o
Outcome:  X Ganhou

Linha 9 Tabuleiro:
  | x | x
  | x | o
x | o | o
Outcome:  X Ganhou

Linha 10 Tabuleiro:
x |   |  
o | x | o
x | o | x
Outcome:  X Ganhou

Linha 11 Tabuleiro:
x | o | x
x | o | o
x |   |  
Outcome:  X Ganhou

Linha 12 Tabuleiro:
x | x | x
  |   | o
  |   | o
Outcome:  X Ganhou

Linha 13 Tabuleiro:
x |   | o
o | x |  
x | o | x
Outcome:  X Ganhou

Linha 14 Tabuleiro:
  | x | o


In [177]:
input = [
    -1,  1, -1,
     0,  0,  1,
     1,  1, -1,
]
input = {columnNames[i]: input[i] for i in range(len(input))}
input = pd.DataFrame([input])
outcome = kNN_classifier.predict(input)
print(outcome)

['O Ganhou']
